# 2. Fine-Tuning BERT on SQuAD

This is the core notebook — fine-tuning a transformer for extractive question answering.

**Hardware note:** I'm running this on a laptop with no GPU and limited disk space, so a
few deliberate choices are made below to keep this feasible on CPU:

- **Model:** `distilbert-base-uncased` instead of `bert-base-uncased`. DistilBERT keeps
  BERT's architecture/behavior but is ~40% smaller and faster, which matters a lot when
  there's no GPU. Swapping to `bert-base-uncased` (or any other BERT variant) is a
  one-line change if you have a GPU (e.g. free Colab) and want closer-to-paper results.
- **Data subset:** instead of the full ~87k training examples, we train on a few
  thousand. This is enough to see the model actually learn the task in a reasonable
  amount of time on CPU.
- **1 epoch, small batch size:** kept intentionally light. The goal of this notebook is
  to demonstrate the full fine-tuning pipeline correctly, not to chase SQuAD leaderboard
  numbers on hardware that was never meant for this.

If you want to scale this up: increase `NUM_TRAIN_EXAMPLES`, `NUM_EPOCHS`, and change
`MODEL_NAME` to `"bert-base-uncased"`, then run this on a GPU (Colab's free tier works).

In [1]:
# !pip install -q transformers datasets torch evaluate
# (uncomment the line above if these aren't already installed in your environment)

In [2]:
import os
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    default_data_collator,
)

MODEL_NAME = "distilbert-base-uncased"   # swap to "bert-base-uncased" if you have a GPU
NUM_TRAIN_EXAMPLES = 3000                # subset size — CPU friendly
NUM_EVAL_EXAMPLES = 500
MAX_LENGTH = 384                         # max tokens per context+question
DOC_STRIDE = 128                         # overlap when a context is split into chunks
NUM_EPOCHS = 1
BATCH_SIZE = 8

OUTPUT_DIR = Path("../models/bert-squad-finetuned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("No GPU detected — that's expected on this setup, training is scaled down accordingly.")

Using device: cpu
No GPU detected — that's expected on this setup, training is scaled down accordingly.


## Load SQuAD via the `datasets` library

We use HuggingFace `datasets` here (rather than re-parsing the raw JSON like notebook 1)
because it gives us a ready-to-use train/validation split, and integrates directly with
the tokenizer and `Trainer` below. We immediately slice down to a small subset to keep
things CPU-friendly.

In [3]:
# Use the canonical SQuAD repo ID; the bare "squad" name can trigger an invalid HF URI in some environments.
raw_datasets = load_dataset("rajpurkar/squad")

train_dataset = raw_datasets["train"].shuffle(seed=42).select(range(NUM_TRAIN_EXAMPLES))
eval_dataset = raw_datasets["validation"].shuffle(seed=42).select(range(NUM_EVAL_EXAMPLES))

print(train_dataset)
print(eval_dataset)
train_dataset[0]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 3000
})
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 500
})


{'id': '573173d8497a881900248f0c',
 'title': 'Egypt',
 'context': 'The Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for religious freedom. The United States Commission on International Religious Freedom, a bipartisan independent agency of the US government, has placed Egypt on its watch list of countries that require close monitoring due to the nature and extent of violations of religious freedom engaged in or tolerated by the government. According to a 2010 Pew Global Attitudes survey, 84% of Egyptians polled supported the death penalty for those who leave Islam; 77% supported whippings and cutting off of hands for theft and robbery; and 82% support stoning a person who commits adultery.',
 'question': 'What percentage of Egyptians polled support death penalty for those leaving Islam?',
 'answers': {'text': ['84%'], 'answer_start': [468]}}

## Tokenize and align answer spans to token positions

This is the trickiest part of QA fine-tuning: SQuAD gives us the answer as a **character
offset** in the raw context, but the model needs the answer as a **token index**. We use
the tokenizer's `offset_mapping` to convert one to the other.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def prepare_train_features(examples):
    # Tokenize question + context together. If the context is too long it gets split
    # into overlapping chunks (doc_stride), so one example can produce multiple features.
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = tokenized.sequence_ids(i)
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        if len(answers["answer_start"]) == 0:
            # No answer for this example (not expected for SQuAD v1.1, but handled just in case)
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        # Find where the context starts/ends within this tokenized chunk
        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1
        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        # If the answer isn't fully inside this chunk, label it with the CLS index
        if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                token_start_index += 1
            start_positions.append(token_start_index - 1)

            while offsets[token_end_index][1] >= end_char:
                token_end_index -= 1
            end_positions.append(token_end_index + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized


tokenized_train = train_dataset.map(
    prepare_train_features, batched=True, remove_columns=train_dataset.column_names
)
tokenized_eval = eval_dataset.map(
    prepare_train_features, batched=True, remove_columns=eval_dataset.column_names
)

print(f"Tokenized train features: {len(tokenized_train)}")
print(f"Tokenized eval features:  {len(tokenized_eval)}")

Tokenized train features: 3028
Tokenized eval features:  506


## Load the pretrained model and add a QA head

In [5]:
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
model.to(device)
print(f"Loaded {MODEL_NAME} with {model.num_parameters():,} parameters")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded distilbert-base-uncased with 66,364,418 parameters


## Set up training

Small batch size, single epoch, no mixed precision (fp16 needs a GPU) — kept
conservative so this actually finishes on a CPU-only machine in a reasonable time.

In [6]:
import importlib
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from packaging.version import parse

# Install with the exact Python interpreter used by this notebook kernel.
try:
    acc_version = version("accelerate")
    needs_install = parse(acc_version) < parse("1.1.0")
except PackageNotFoundError:
    needs_install = True
except Exception:
    needs_install = True

if needs_install:
    print("Installing accelerate>=1.1.0 using the active notebook kernel...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "accelerate>=1.1.0"])

# Re-import after installation so the current kernel sees the fresh package.
import accelerate

training_args = TrainingArguments(
    output_dir="../models/checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    fp16=False,          # no GPU here, so no mixed precision
    logging_steps=25,
    save_total_limit=1,  # keep disk usage down — only keep the latest checkpoint
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=default_data_collator,
    processing_class=tokenizer,
)

In [7]:
train_result = trainer.train()
print(train_result)

c:\Users\user\Desktop\bert-qa-squad\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,2.814596,2.542461


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=379, training_loss=3.4556093341756937, metrics={'train_runtime': 5816.8364, 'train_samples_per_second': 0.521, 'train_steps_per_second': 0.065, 'total_flos': 296713190172672.0, 'train_loss': 3.4556093341756937, 'epoch': 1.0})


## Save the fine-tuned model

This is what `app.py` (the Flask demo) looks for at `models/bert-squad-finetuned/`.
If this folder is missing or empty, the web app automatically falls back to a
pretrained SQuAD checkpoint from the HuggingFace Hub instead.

In [8]:
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

saved_files = list(OUTPUT_DIR.glob("*"))
total_size_mb = sum(f.stat().st_size for f in saved_files if f.is_file()) / 1e6
print(f"Saved {len(saved_files)} files to {OUTPUT_DIR} (~{total_size_mb:.1f} MB total)")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved 5 files to ..\models\bert-squad-finetuned (~266.2 MB total)


## Optional: clean up training checkpoints to save disk space

`TrainingArguments` above already saves at most 1 checkpoint, but if disk space is
tight, this removes the intermediate checkpoint folder entirely once we've already
saved the final model above.

In [9]:
import shutil

checkpoint_dir = Path("../models/checkpoints")
if checkpoint_dir.exists():
    size_mb = sum(f.stat().st_size for f in checkpoint_dir.rglob("*") if f.is_file()) / 1e6
    shutil.rmtree(checkpoint_dir)
    print(f"Removed {checkpoint_dir} and freed ~{size_mb:.1f} MB")
else:
    print("No checkpoint directory to clean up.")

Removed ..\models\checkpoints and freed ~797.2 MB


## Next step

Head over to `3_Model_Evaluation.ipynb` to properly evaluate this model with
Exact Match / F1 and look at some real predictions side-by-side with ground truth.